# Examen Final — Sistema de Recuperación de Información con RAG sobre arXiv Paper Abstracts

**Curso:** ICCD753 Recuperación de Información 2026-A — Prof. Iván Carrera, EPN-FIS
**Corpus:** [arXiv Paper Abstracts (Kaggle)](https://www.kaggle.com/datasets/spsayakpaul/arxiv-paper-abstracts)

Este notebook documenta e implementa un sistema RAG (Retrieval-Augmented Generation) completo:
búsqueda semántica sobre una base de datos vectorial de abstracts de papers de arXiv, generación
de respuestas con un LLM (Google Gemini) fundamentadas en los documentos recuperados, presentación
de evidencias, y evaluación del sistema.

Toda la lógica está implementada en el paquete `backend/` (para que la misma implementación sirva
como servicio detrás de la interfaz web de chat); este notebook la importa, la ejecuta y documenta
cada requerimiento del enunciado (`Guia del examen/examen2bim.pdf`), sección por sección.

**URL del chat RAG desplegado:** `TODO: pegar aquí la URL pública una vez desplegado (ver DEPLOY.md)`


In [ ]:
# Si ejecutas este notebook en un entorno nuevo (Colab, EC2, etc.), instala las dependencias:
# %pip install -q -r backend/requirements.txt

import sys, os
sys.path.append(os.path.abspath(".."))  # no-op si ya corres desde la raíz del repo
os.environ.setdefault("GEMINI_API_KEY", "")  # exporta tu clave real como variable de entorno antes de correr esto

from backend import corpus, embeddings, vector_db, rag, evaluation


## A. Preparación del corpus

El corpus se descarga desde Kaggle (`spsayakpaul/arxiv-paper-abstracts`, columnas `titles`,
`summaries`, `terms`) usando `kagglehub` (no requiere credenciales manuales de la API de Kaggle
en la mayoría de entornos). Se toma un subconjunto de ~4000 papers para que la indexación sea
rápida, y se construyen dos artefactos:

- `backend/data/corpus.json`: `paper_id`, `title`, `abstract`, `categories` (los tópicos/terms de arXiv).
- `backend/data/qrels.json`: juicios de relevancia sintéticos para la evaluación (Sección I), ya que
  el dataset crudo no trae pares consulta-relevancia. Cada una de las 10 consultas de referencia se
  mapea a las categorías de arXiv (`cs.CL`, `cs.CV`, `cs.LG`, ...) que se consideran relevantes,
  siguiendo el mismo enfoque de qrels-por-categoría usado en un proyecto previo del curso.

Si la descarga falla (sin conexión, límite de Kaggle, etc.) se genera automáticamente un corpus de
prueba reducido para que el resto del pipeline siga siendo ejecutable.


In [ ]:
corpus.download_and_prepare_corpus()

import json
with open(corpus.CORPUS_PATH, encoding="utf-8") as f:
    papers = json.load(f)

print(f"Papers en el corpus: {len(papers)}")
papers[0]


## B. Representación mediante embeddings

Cada documento se representa concatenando `title` + `abstract` y codificándolo con un modelo de
embeddings de oraciones (`sentence-transformers/all-MiniLM-L6-v2`), el mismo modelo usado en varios
ejercicios del curso. Los embeddings se normalizan (`normalize_embeddings=True`) para que el
producto interno equivalga a similitud coseno.


In [ ]:
sample_texts = [f"{p['title']}. {p['abstract']}" for p in papers[:3]]
sample_vectors = embeddings.encode_text(sample_texts)

print("Dimensión del embedding:", sample_vectors.shape[1])
print("Norma L2 del primer vector (debe ser ~1.0 por la normalización):", float((sample_vectors[0] ** 2).sum() ** 0.5))


## C. Almacenamiento y búsqueda vectorial

Los embeddings de todo el corpus se indexan en **FAISS** (`IndexFlatIP`, búsqueda exacta por
producto interno = similitud coseno sobre vectores normalizados). El índice y el mapeo
`índice → paper_id` se persisten en `backend/data/faiss.index` para no tener que recalcular los
embeddings en cada arranque del servicio.


In [ ]:
vector_db.build_index()
print("Índice FAISS construido y guardado en:", vector_db.INDEX_PATH)


## D. Recuperación

Dada una consulta en lenguaje natural, se codifica con el mismo modelo de embeddings y se buscan
los `top_k` documentos más cercanos en el índice FAISS. A continuación se muestran los resultados
de recuperación para las consultas de ejemplo del enunciado del examen.


In [ ]:
import pandas as pd

example_queries = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
]

rows = []
for q in example_queries:
    for r in vector_db.search(q, top_k=3):
        rows.append({"Consulta": q, "Documento": r["title"], "Categorías": ", ".join(r["categories"]), "Similitud": round(r["similarity"], 3)})

pd.DataFrame(rows)


## E. Generación aumentada por recuperación

`backend/rag.py` implementa el pipeline completo:

1. Búsqueda vectorial (top-10) sobre el índice FAISS.
2. **Re-ranking** con un cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`, el mismo modelo usado
   en el Ejercicio 10 de re-ranking del curso) que reordena los candidatos por relevancia real
   consulta-documento, quedándose con los 4 mejores.
3. Construcción de un prompt que incluye los documentos recuperados como contexto.
4. Generación de la respuesta final con **Google Gemini** (`gemini-2.5-flash`), instruido a responder
   ÚNICAMENTE con base en el contexto recuperado y a declarar explícitamente cuando la evidencia sea
   insuficiente (ver Sección F).

Requiere la variable de entorno `GEMINI_API_KEY` (nunca hardcodeada en el código ni en el repositorio).


In [ ]:
result = rag.generate_rag_response(example_queries[0])
print(result["answer"])


## F. Presentación de evidencias

Cada respuesta va acompañada de la lista de documentos (`evidences`) usados como contexto —
título, abstract, categorías y similitud coseno — de modo que se pueda verificar la relación entre
la consulta, los documentos recuperados y la respuesta generada. Cuando la mejor similitud
recuperada cae por debajo de un umbral (`INSUFFICIENT_EVIDENCE_THRESHOLD = 0.30`), el campo
`insufficient_evidence` se marca en `True` y el prompt instruye al LLM a decirlo explícitamente en
vez de responder con evidencia débil o inventada. La interfaz web muestra esto mismo como un aviso
visible y como un acordeón de evidencias debajo de cada respuesta.


In [ ]:
for e in result["evidences"]:
    print(f"- [{e['similarity']:.3f}] {e['title']}  (categorías: {', '.join(e['categories'])})")

print("\n¿Evidencia insuficiente para esta consulta?:", result["insufficient_evidence"])

# Ejemplo de una consulta fuera del dominio del corpus, para mostrar el caso de evidencia insuficiente
out_of_domain = rag.generate_rag_response("What is the best recipe for a chocolate cake?")
print("\nRespuesta a consulta fuera de dominio:\n", out_of_domain["answer"][:400])
print("¿Evidencia insuficiente?:", out_of_domain["insufficient_evidence"])


## G. Interfaz web conversacional

La interfaz de chat está implementada en Next.js/React (carpeta `src/` de este mismo repositorio:
`src/app/page.tsx`, `src/components/chat-area.tsx`). Permite:

- Ingresar consultas en lenguaje natural desde un cuadro de texto tipo chat.
- Visualizar la respuesta generada, el aviso de "evidencia insuficiente" cuando aplica, y un
  acordeón con las evidencias (papers) recuperadas para esa respuesta.
- Enviar nuevas consultas sin recargar la página; cada consulta se procesa de forma independiente
  (no se implementa memoria conversacional, tal como permite el enunciado).

El front llama al backend FastAPI (`backend/main.py`, endpoint `POST /api/chat`) a través de la
URL configurada en la variable de entorno `NEXT_PUBLIC_API_URL`.

Para correrlo localmente:

```bash
# Backend
source backend/.venv/bin/activate
export GEMINI_API_KEY="tu-api-key"
python -m uvicorn backend.main:app --reload --port 8000

# Frontend (en otra terminal)
npm install
npm run dev
```


## H. Despliegue en la nube

El sistema se despliega en una instancia **AWS EC2** (capa gratuita, `t3.micro`/`t2.micro`), donde
`nginx` enruta por path a los dos procesos que corren en la misma máquina:

- `/api/*` → backend FastAPI (`uvicorn`, puerto 8000 interno).
- `/*` → frontend Next.js (`next start`, puerto 3000 interno).

Los pasos detallados de aprovisionamiento (paquetes, `systemd`, `nginx`, variables de entorno,
swap para la RAM limitada de un `t2.micro`/`t3.micro`) están en [`DEPLOY.md`](./DEPLOY.md) de este
repositorio. La clave `GEMINI_API_KEY` se exporta como variable de entorno del servicio systemd del
backend — nunca se guarda en el código fuente ni se sube al repositorio (está además excluida vía
`.gitignore` / `.env`).

**URL pública del sistema desplegado:** `TODO: completar con la URL/IP de la instancia EC2 una vez desplegada`


## I. Evaluación del sistema y de la generación

### I.1 Métricas de recuperación

Se calculan Precision@k, Recall@k y NDCG@k (k = 1, 3, 5) contra los juicios de relevancia sintéticos
de `backend/data/qrels.json` (Sección A), reutilizando el mismo patrón de evaluación de proyectos
previos del curso.


In [ ]:
report = evaluation.run_evaluation()
pd.DataFrame(report).T


### I.2 Juicio subjetivo sobre la generación

Para un conjunto de consultas de prueba (incluyendo casos dentro y fuera del dominio del corpus),
se evalúa manualmente cada respuesta según los 5 criterios pedidos en el enunciado. Completa esta
tabla ejecutando el sistema con tus propias consultas y registrando tu apreciación:

| Consulta | Corrección | Relevancia | Fidelidad a evidencias | Integra varios documentos | Reconoce info. insuficiente |
|---|---|---|---|---|---|
| "What are the main applications of Graph Neural Networks?" | ✅/⚠️/❌ | ✅/⚠️/❌ | ✅/⚠️/❌ | ✅/⚠️/❌ | N/A |
| "How is reinforcement learning used in robotics?" | | | | | |
| "Recent advances in diffusion models for image generation." | | | | | |
| "Techniques for improving retrieval-augmented generation systems." | | | | | |
| "What is the best recipe for a chocolate cake?" (fuera de dominio) | N/A | N/A | N/A | N/A | ✅/⚠️/❌ |

**Conclusiones técnicas:**

- *Corrección de la respuesta*: TODO — completar tras revisar manualmente las respuestas generadas.
- *Relevancia respecto a la consulta*: TODO.
- *Fidelidad respecto de las evidencias recuperadas*: TODO — verificar que la respuesta no contenga
  afirmaciones que no estén sustentadas en los abstracts mostrados como evidencia.
- *Capacidad de integrar información de varios documentos*: TODO — revisar si las respuestas citan
  y combinan más de un `[Documento N]` cuando corresponde.
- *Capacidad de reconocer cuando el corpus no tiene información suficiente*: TODO — confirmar con el
  ejemplo fuera de dominio de la Sección F que el sistema lo declara explícitamente en vez de alucinar.

**Propuestas de mejora futuras:** TODO (p. ej. aumentar el tamaño del subconjunto del corpus,
comparar otros modelos de embeddings, agregar métricas automáticas de fidelidad/faithfulness con un
LLM juez, etc.)
